<a href="https://colab.research.google.com/github/elviakiran-miranda-hue/BUS4118S26/blob/dev/Prompt_eng_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json

class TravelSupportAI:
    def __init__(self):
        # Define issue categories and their required fields
        self.categories = {
            "CANCELLATION": ["booking_id", "reason"],
            "REFUND": ["booking_id", "original_payment_method"],
            "FLIGHT_CHANGE": ["booking_id", "new_date"],
            "GENERAL_QUERY": []
        }

        # High-risk keywords for immediate escalation
        self.escalation_triggers = ["lawsuit", "legal", "emergency", "injury", "hospital"]

    def classify_intent(self, user_input):
        """Mock LLM: Identifies the category of the issue."""
        user_input = user_input.lower()
        if "cancel" in user_input: return "CANCELLATION"
        if "refund" in user_input: return "REFUND"
        if "change" in user_input or "rebook" in user_input: return "FLIGHT_CHANGE"
        return "GENERAL_QUERY"

    def extract_entities(self, user_input):
        """Mock LLM: Extracts specific data points from text."""
        # In a real app, use Regex or an NER model here
        words = user_input.split()
        entities = {}
        for word in words:
            if word.startswith("BK-"): # Example Booking ID format
                entities["booking_id"] = word
        return entities

    def handle_request(self, user_text):
        # 1. Immediate Escalation Check (Safety/Legal)
        if any(trigger in user_text.lower() for trigger in self.escalation_triggers):
            return "ESCALATE: This request involves a sensitive matter and is being sent to a human supervisor immediately."

        # 2. Intent & Data Extraction
        intent = self.classify_intent(user_text)
        provided_info = self.extract_entities(user_text)

        # 3. Gap Analysis (Gather missing info)
        required_fields = self.categories.get(intent, [])
        missing = [field for field in required_fields if field not in provided_info]

        if missing:
            readable_missing = ", ".join(missing).replace("_", " ")
            return f"I can help with your {intent.lower()}. To proceed, I just need your: {readable_missing}."

        # 4. Propose Solution
        return self.get_solution(intent, provided_info)

    def get_solution(self, intent, info):
        """Returns a policy-based solution."""
        solutions = {
            "CANCELLATION": f"I've located booking {info.get('booking_id')}. Per our 24h policy, you are eligible for a full credit. Should I process this?",
            "REFUND": f"Refund request for {info.get('booking_id')} received. Expect a status update within 3-5 business days.",
            "FLIGHT_CHANGE": "I can help change your flight. Please provide your preferred new date.",
            "GENERAL_QUERY": "How else can I assist with your travel plans today?"
        }
        return solutions.get(intent, "How can I help you today?")

# --- Example Usage ---
bot = TravelSupportAI()

# Scenario A: Missing Info
print(f"User: I want to cancel my trip.\nAI: {bot.handle_request('I want to cancel my trip.')}\n")

# Scenario B: Full Info Provided
print(f"User: Cancel BK-9922 because of a schedule conflict.\nAI: {bot.handle_request('Cancel BK-9922 because of a schedule conflict.')}\n")

# Scenario C: Escalation
print(f"User: My flight was a disaster and I'm calling my lawyer!\nAI: {bot.handle_request('My flight was a disaster and I am calling my lawyer!')}")

User: I want to cancel my trip.
AI: I can help with your cancellation. To proceed, I just need your: booking id, reason.

User: Cancel BK-9922 because of a schedule conflict.
AI: I can help with your cancellation. To proceed, I just need your: reason.

User: My flight was a disaster and I'm calling my lawyer!
AI: How else can I assist with your travel plans today?
